# Illustrate the Enemy Formations

In [9]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:70% !important; }</style>"))

### Read in the formation data

In [69]:
"""
; The first six bytes select the movement strategy
; for the formation from enemyMovementStrategyLoPtrArray.
.BYTE $01 ; Movement Strategy: Enemy 1
.BYTE $01 ; Movement Strategy: Enemy 2
.BYTE $01 ; Movement Strategy: Enemy 3
.BYTE $01 ; Movement Strategy: Enemy 4
.BYTE $01 ; Movement Strategy: Enemy 5
.BYTE $00 ; Movement Strategy: Enemy 6
; The next six bytes select the inital Y position for
; each enemy.
.BYTE $9C ; Initial Y Position : Enemy 1
.BYTE $6C ; Initial Y Position : Enemy 2
.BYTE $B4 ; Initial Y Position : Enemy 3
.BYTE $84 ; Initial Y Position : Enemy 4
.BYTE $CC ; Initial Y Position : Enemy 5
.BYTE $00 ; Initial Y Position : Enemy 6
; THe delay before adding the next enemy, think of it as
; the spacing between enemies as they enter the screen.
.BYTE $05 ; Delay between spawning enemies.
.BYTE $00 ; Initial X Position of Enemy 1
; The last byte is the sprite to be used for the enemies.
.BYTE $09 ; Sprite Value for Enemies
.BYTE $00 ; End Sentinel
"""
game_data_file = "formation_data.asm"
lines_in_file = open(game_data_file,'r').readlines()

formations = []
strategies, y_positions = [],[]
for l in lines_in_file:
    if l.startswith("enemy") and strategies:
        formations += [(
            strategies,
            y_positions,
            delay,
            initial_x,
            sprite_value
        )]
        strategies, y_positions = [],[]
    if "BYTE" not in l:
        continue
    v = int(l[15:17],16)
    if "Movement Strategy" in l:
        strategies += [v]
    elif "Y Position" in l:
        y_positions += [v]
    elif "Delay" in l:
        delay = v
    elif "Initial X" in l:
        initial_x = v
    elif "Sprite Value" in l:
        sprite_value = "{:02X}".format(v)

formations += [(
    strategies,
    y_positions,
    delay,
    initial_x,
    sprite_value
)]


In [65]:
mkdir -p formations

In [74]:
from PIL import Image, ImageDraw, ImageFont
from itertools import pairwise

double_size = lambda img: img.resize((int(img.width * 2), int(img.height * 2)), Image.NEAREST)

for i,formation in enumerate(formations):
    _, y_positions, delay, initial_x, sprite_value = formation
    sprite = Image.open(f"meanie_sprites/MEANIE_{sprite_value}.png")
    sprite = double_size(sprite)

    img = Image.new('RGBA', (500 , 500))
    x = 200
    py = 0
    for y,ny in pairwise(y_positions):
        if not y: continue
        img.paste(sprite, (x*2,y*2))
        x -= delay * 2 if ny != y else int(sprite.width/2)
    img.save(f"formations/formation_{i}.png")
